# 14 (DE) — Ingestion & Bulk Load

**Data Engineer perspective.** Getting data *into* IRIS at scale: chunked multi-row inserts from pandas, type inference, server-side `LOAD DATA`, local vs foreign file reads, and what happens when a chunk fails mid-stream.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Bulk load from pandas — 10,000 rows

`createDataFrame` sends rows in adaptive chunks using IRIS's supported multi-row form (`INSERT ... SELECT ... UNION ALL ...`). Chunk size is capped by row count *and* by statement size (IRIS's SQL preparer rejects some very large UNION ALL chains), so one round-trip carries as many rows as is safe.

In [ ]:
import time
import pandas as pd

N = 10_000
big = pd.DataFrame({
    "evento_id": range(1, N + 1),
    "sensor": [f"s{i % 40}" for i in range(1, N + 1)],
    "temperatura": [round(20 + (i % 15) * 0.7, 2) for i in range(1, N + 1)],
})

t0 = time.perf_counter()
eventos = session.createDataFrame(big)
t1 = time.perf_counter()
print(f"ingested {eventos.count()} rows in {t1 - t0:.2f}s")

## 2. Type inference

Column types are inferred from a sample of the data.

In [ ]:
eventos.printSchema()

## 3. Dict rows are also accepted

In [ ]:
session.createDataFrame([
    {"evento_id": 10001, "sensor": "sx", "temperatura": 21.5},
    {"evento_id": 10002, "sensor": "sy", "temperatura": 22.5},
]).show()

## 4. Server-side LOAD DATA

`read.load_data` runs IRIS's native loader — but the path must exist **on the IRIS server filesystem**, not on the client. Attempting a client-side path shows the contract.

In [ ]:
import tempfile, os

tmp = tempfile.mkdtemp()
csv_path = os.path.join(tmp, "eventos.csv")
big.head(100).to_csv(csv_path, index=False)

try:
    session.read.load_data(csv_path, "eventos_load", format="csv")
except Exception as e:
    print("load_data contract:", str(e)[:160])

## 5. Local file reads (client-side Arrow)

In [ ]:
parquet_path = os.path.join(tmp, "eventos.parquet")
big.head(50).to_parquet(parquet_path, index=False)

session.read.csv(csv_path).show(3)
session.read.parquet(parquet_path).show(3)

## 6. Foreign-table read (IRIS owns the file)

With `foreign=True`, IRIS reads the file directly; rows stream to Python only on actions. `server_path` tells IRIS where the shared volume lives on its side (`/irispark-data` in compose).

In [ ]:
import os

foreign_dir = "data/foreign_demo"
os.makedirs(foreign_dir, exist_ok=True)
big.head(100).to_csv(os.path.join(foreign_dir, "eventos.csv"), index=False)

try:
    session.read.csv(
        os.path.join(foreign_dir, "eventos.csv"),
        foreign=True,
        server_path="/irispark-data/foreign_demo",
        options={"header": True},
    ).show(3)
except Exception as e:
    print("foreign read not available from this path:", str(e)[:120])

## 7. Failure contract — no silent retries

If a chunk fails mid-stream, ingestion raises immediately naming the failed row range. Earlier chunks stay committed; nothing is re-sent automatically (that would duplicate rows). See `tests/test_batch_insert.py` for the pinned behavior.

**Operational rule**: wrap bulk loads in your orchestrator's idempotency pattern (delete-by-batch-id + re-run, or staging table + swap).

In [ ]:
print("failure contract: RuntimeError('bulk insert into ... failed for rows [i..j] of N')")

## 8. Cleanup

In [ ]:
for t in ("eventos_load",):
    session.sql(f"DROP TABLE IF EXISTS {t}")
print("cleaned up")

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")